# Gaussian: Smoothing minus RobustCE at matched strengths

## 1. Comparison

Directly compare the fitted models from notebooks 03 and 04 on **ordinary clean test CE**, with the analytical pairing
$$
\varepsilon=2\delta.
$$
For each eta, delta, and paired evaluation seed, compute
$$
D_s(\eta,\delta)
=\mathrm{CE}_{\mathrm{test}}(\mathrm{Smoothing},\eta,2\delta,s)
-\mathrm{CE}_{\mathrm{test}}(\mathrm{RobustCE},\eta,\delta,s).
$$

- **Negative / red:** smoothing has lower clean test CE.
- **Positive / green:** RobustCE has lower clean test CE.
- **Zero or nearly zero / white:** similar test CE.

This is a difference between the **same evaluation metric for two separately trained models**, not a subtraction of their different training objectives.

Both source notebooks already tuned learning rate separately using clean validation CE. This notebook preserves those rates and reuses their completed training runs. It does not select epsilon or delta using test results and does not train new models.

Run notebooks 03 and 04 first. Text is in English, figures use Plotly, and no tables are displayed.


In [1]:
import ast
import csv
import hashlib
import json
import math
import statistics
from itertools import product
from pathlib import Path

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
RESULTS_ROOT = PROJECT_ROOT / "results" / "05_gaussian_smoothing_vs_robust"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_STEMS = {
    "RobustCE": "03_gaussian_eta_delta",
    "Smoothing": "04_gaussian_eta_epsilon",
}
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"


def read_runs(path):
    records = []
    with path.open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            record = {}
            for key, value in row.items():
                if key in ("status", "error"):
                    record[key] = value
                elif value == "":
                    record[key] = None
                elif key == "seed":
                    record[key] = int(value)
                else:
                    record[key] = float(value)
            records.append(record)
    return records


def write_csv(path, records):
    if not records:
        return
    keys = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(records)


def summarize(values):
    return statistics.mean(values), statistics.stdev(values) if len(values) > 1 else 0.0


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 2. Matching epsilon to delta

For observed binary labels and signed margin $m=(2y-1)z$,
$$
L_{\mathrm{LS}}=L_{\mathrm{CE}}+\frac{\varepsilon}{2}m,\qquad
L_{\mathrm{RobustCE}}=L_{\mathrm{CE}}+\delta\operatorname{ReLU}(m).
$$
With epsilon = 2 * delta, the penalties agree on positive margins. On negative margins,
$$
L_{\mathrm{RobustCE}}-L_{\mathrm{LS}}
=\delta\operatorname{ReLU}(-m).
$$
Thus the pairing aligns the positive-margin penalty while retaining the worst-case construction's treatment of negative margins. It does not make the complete losses or fitted predictions identical.

The saved grids include delta from 0 to 0.5 and corresponding epsilon from 0 to 1. Epsilon is matched by this formula, not tuned against test data.


In [2]:
def load_bundle(stem):
    latest_path = PROJECT_ROOT / "results" / stem / "latest_run.json"
    notebook_path = PROJECT_ROOT / "notebooks" / (stem + ".ipynb")
    if not latest_path.exists() or not notebook_path.exists():
        raise FileNotFoundError(f"Run notebooks/03_gaussian_eta_delta.ipynb and notebooks/04_gaussian_eta_epsilon.ipynb first; missing {stem}.")
    latest = json.loads(latest_path.read_text(encoding="utf-8"))
    directory = Path(latest["run_directory"])
    required = ("settings.json", "protocol.json", "tuning_runs.csv", "evaluation_runs.csv", "selected_settings.json")
    missing = [name for name in required if not (directory / name).exists()]
    if missing:
        raise FileNotFoundError(f"Incomplete source run {stem}: {missing}. Finish that notebook first.")
    settings = json.loads((directory / "settings.json").read_text(encoding="utf-8"))
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    implementation = "\n".join(
        "".join(cell["source"]) for cell in notebook["cells"]
        if cell.get("id") in ("data", "training")
    )
    expected_id = hashlib.sha256(
        json.dumps({"settings": settings, "implementation": implementation}, sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]
    if expected_id != latest["run_id"]:
        raise ValueError(f"{stem} source/settings no longer match its saved run. Rerun that source notebook.")
    return {
        "stem": stem, "run_id": latest["run_id"], "directory": directory,
        "settings": settings,
        "protocol": json.loads((directory / "protocol.json").read_text(encoding="utf-8")),
        "source": {cell.get("id"): "".join(cell["source"]) for cell in notebook["cells"]},
        "tuning": read_runs(directory / "tuning_runs.csv"),
        "evaluation": read_runs(directory / "evaluation_runs.csv"),
        "selected": json.loads((directory / "selected_settings.json").read_text(encoding="utf-8")),
        "file_hashes": {name: sha256(directory / name) for name in required},
    }


sources = {method: load_bundle(stem) for method, stem in SOURCE_STEMS.items()}
robust_source = sources["RobustCE"]
smoothing_source = sources["Smoothing"]
common_keys = (
    "torch_version", "device", "dtype", "threads", "etas", "learning_rates",
    "tuning_seeds", "evaluation_seeds", "split_sizes", "epochs",
    "win_atol", "optimizer", "batching", "selection",
)
for key in common_keys:
    if robust_source["settings"].get(key) != smoothing_source["settings"].get(key):
        raise ValueError(f"Incompatible source settings: {key}.")
if robust_source["source"]["data"] != smoothing_source["source"]["data"]:
    raise ValueError("Source datasets/model initializations differ.")


def canonical_function(source, name):
    node = next(
        item for item in ast.parse(source).body
        if isinstance(item, ast.FunctionDef) and item.name == name
    )

    class NormalizeStrength(ast.NodeTransformer):
        def visit_Name(self, item):
            if item.id in ("delta", "epsilon"):
                item.id = "strength"
            return item

        def visit_arg(self, item):
            if item.arg in ("delta", "epsilon"):
                item.arg = "strength"
            return item

    return ast.dump(NormalizeStrength().visit(node), include_attributes=False)


for name in ("train_one", "evaluate"):
    if canonical_function(robust_source["source"]["training"], name) != canonical_function(smoothing_source["source"]["training"], name):
        raise ValueError(f"Source training/evaluation procedures differ: {name}.")

ETAS = tuple(robust_source["settings"]["etas"])
DELTAS = tuple(robust_source["settings"]["deltas"])
AVAILABLE_EPSILONS = tuple(smoothing_source["settings"]["epsilons"])
TUNING_SEEDS = tuple(robust_source["settings"]["tuning_seeds"])
EVALUATION_SEEDS = tuple(robust_source["settings"]["evaluation_seeds"])
LEARNING_RATES = tuple(robust_source["settings"]["learning_rates"])
WIN_ATOL = robust_source["settings"]["win_atol"]
assert set(TUNING_SEEDS).isdisjoint(EVALUATION_SEEDS)
PAIRS = []
for delta in DELTAS:
    matches = [
        epsilon for epsilon in AVAILABLE_EPSILONS
        if math.isclose(epsilon, 2 * delta, rel_tol=0, abs_tol=1e-12)
    ]
    if len(matches) != 1:
        raise ValueError(f"Need exactly one epsilon=2*delta for delta={delta}, found {matches}.")
    PAIRS.append((delta, matches[0]))

print(f"Verified matching protocols: {len(ETAS)} eta values, {len(PAIRS)} matched strength pairs, {len(EVALUATION_SEEDS)} paired seeds.")
print("No new training: both methods retain their own validation-selected learning rates.")


Verified matching protocols: 7 eta values, 12 matched strength pairs, 5 paired seeds.
No new training: both methods retain their own validation-selected learning rates.


In [3]:
strength_names = {"RobustCE": "delta", "Smoothing": "epsilon"}
selected = {}
evaluation_index = {}
for method, source in sources.items():
    strength_name = strength_names[method]
    strengths = source["settings"][strength_name + "s"]
    tuning_index = {
        (r["seed"], r["eta"], r[strength_name], r["lr"]): r
        for r in source["tuning"]
    }
    if len(tuning_index) != len(source["tuning"]):
        raise ValueError(f"Duplicate tuning records for {method}.")
    configs = {(r["eta"], r[strength_name]): r["lr"] for r in source["selected"]}
    for eta, strength in product(ETAS, strengths):
        candidates = []
        for lr in LEARNING_RATES:
            records = [tuning_index.get((seed, eta, strength, lr)) for seed in TUNING_SEEDS]
            if all(row is not None and row["status"] == "ok" for row in records):
                candidates.append((statistics.mean(row["validation_ce"] for row in records), lr))
        if not candidates or (eta, strength) not in configs:
            raise ValueError(f"No complete selected configuration for {method}, eta={eta}, strength={strength}.")
        if min(candidates)[1] != configs[(eta, strength)]:
            raise ValueError(f"Saved lr does not match validation-only selection for {method}, eta={eta}, strength={strength}.")
    selected[method] = configs
    index = {}
    for row in source["evaluation"]:
        key = row["seed"], row["eta"], row[strength_name]
        if key in index:
            raise ValueError(f"Duplicate evaluation record: {method}, {key}.")
        if row["lr"] != configs[(row["eta"], row[strength_name])]:
            raise ValueError(f"Evaluation lr differs from frozen selection: {method}, {key}.")
        index[key] = row
    evaluation_index[method] = index

# Verify the loss-matching identity independently at several logits and labels.
z = torch.tensor([-6.0, -2.0, 0.0, 2.0, 6.0] * 2, dtype=torch.float64, device="cpu")
y = torch.tensor([0.0] * 5 + [1.0] * 5, dtype=torch.float64, device="cpu")
m = (2 * y - 1) * z
nominal = F.softplus(z) - y * z
for delta, epsilon in PAIRS:
    robust = nominal + delta * F.relu(m)
    soft_y = (1 - epsilon) * y + epsilon / 2
    smoothing = F.softplus(z) - soft_y * z
    torch.testing.assert_close(robust - smoothing, delta * F.relu(-m), rtol=1e-10, atol=1e-12)
print("Verified independent validation-only lr choices and the matched-strength loss identity.")


Verified independent validation-only lr choices and the matched-strength loss identity.


## 3. Paired clean-test CE differences

Each cell averages five differences using the same eta, seed, dataset, initialization, and observed training labels. The selected learning rates may differ between methods.

A smoothing win means mean difference < -0.000001; a RobustCE win means mean difference > +0.000001. Differences within that numerical tolerance are ties. Record the paired sample standard deviation and seed-level win counts as well as the mean.

Cells without all five valid pairs are marked missing. At delta = epsilon = 0, both sources must give exactly the same CE baseline.


In [4]:
paired_records = []
cell_records = []
for eta, (delta, epsilon) in product(ETAS, PAIRS):
    differences, robust_rows, smoothing_rows = [], [], []
    for seed in EVALUATION_SEEDS:
        robust = evaluation_index["RobustCE"].get((seed, eta, delta))
        smoothing = evaluation_index["Smoothing"].get((seed, eta, epsilon))
        if robust is None or smoothing is None or robust["status"] != "ok" or smoothing["status"] != "ok":
            continue
        if robust["realized_corruption"] != smoothing["realized_corruption"]:
            raise ValueError(f"Unmatched corruption masks: seed={seed}, eta={eta}.")
        difference = smoothing["test_ce"] - robust["test_ce"]
        if delta == 0 and epsilon == 0:
            assert difference == 0
        differences.append(difference)
        robust_rows.append(robust)
        smoothing_rows.append(smoothing)
        paired_records.append(dict(
            seed=seed, eta=eta, delta=delta, epsilon=epsilon,
            realized_corruption=robust["realized_corruption"],
            smoothing_lr=smoothing["lr"], robust_lr=robust["lr"],
            smoothing_test_ce=smoothing["test_ce"], robust_test_ce=robust["test_ce"],
            difference_smoothing_minus_robust=difference,
            smoothing_gain=-difference,
            smoothing_win=difference < -WIN_ATOL, robust_win=difference > WIN_ATOL,
        ))
    complete = len(differences) == len(EVALUATION_SEEDS)
    record = dict(
        eta=eta, delta=delta, epsilon=epsilon, complete=complete, paired_seeds=len(differences),
        smoothing_lr=selected["Smoothing"][(eta, epsilon)],
        robust_lr=selected["RobustCE"][(eta, delta)],
    )
    if complete:
        mean_difference, std_difference = summarize(differences)
        smoothing_mean = statistics.mean(row["test_ce"] for row in smoothing_rows)
        robust_mean = statistics.mean(row["test_ce"] for row in robust_rows)
        winner = "Smoothing" if mean_difference < -WIN_ATOL else "RobustCE" if mean_difference > WIN_ATOL else "tie"
        record.update(
            smoothing_test_ce_mean=smoothing_mean, robust_test_ce_mean=robust_mean,
            difference_mean=mean_difference, difference_std=std_difference,
            relative_difference_percent=100 * mean_difference / robust_mean,
            smoothing_gain_mean=-mean_difference, winner=winner,
            smoothing_seed_wins=sum(value < -WIN_ATOL for value in differences),
            robust_seed_wins=sum(value > WIN_ATOL for value in differences),
            seed_ties=sum(abs(value) <= WIN_ATOL for value in differences),
            smoothing_accuracy_mean=statistics.mean(row["test_accuracy"] for row in smoothing_rows),
            robust_accuracy_mean=statistics.mean(row["test_accuracy"] for row in robust_rows),
            smoothing_confidence_mean=statistics.mean(row["test_confidence"] for row in smoothing_rows),
            robust_confidence_mean=statistics.mean(row["test_confidence"] for row in robust_rows),
        )
        assert math.isclose(mean_difference, smoothing_mean - robust_mean, rel_tol=0, abs_tol=1e-14)
    cell_records.append(record)

complete_cells = [row for row in cell_records if row["complete"]]
counts = {name: sum(row["winner"] == name for row in complete_cells) for name in ("Smoothing", "RobustCE", "tie")}
print(f"Complete cells: {len(complete_cells)}/{len(ETAS)*len(PAIRS)}.")
print(f"Lower mean clean test CE: Smoothing {counts['Smoothing']} cells; RobustCE {counts['RobustCE']} cells; ties {counts['tie']}.")
print("These are observed paired mean differences; the cell counts are not significance tests.")


Complete cells: 84/84.
Lower mean clean test CE: Smoothing 69 cells; RobustCE 6 cells; ties 9.
These are observed paired mean differences; the cell counts are not significance tests.


In [5]:
provenance = {
    method: {
        "notebook": source["stem"] + ".ipynb", "run_id": source["run_id"],
        "directory": str(source["directory"]), "file_hashes": source["file_hashes"],
    }
    for method, source in sources.items()
}
comparison_settings = {
    "sources": provenance, "etas": list(ETAS),
    "matched_pairs": [{"delta": delta, "epsilon": epsilon} for delta, epsilon in PAIRS],
    "tuning_seeds": list(TUNING_SEEDS), "evaluation_seeds": list(EVALUATION_SEEDS),
    "win_atol": WIN_ATOL,
    "difference": "ordinary clean test CE(Smoothing) minus ordinary clean test CE(RobustCE)",
    "matching": "epsilon=2*delta; no test-based strength selection",
}
run_id = hashlib.sha256(json.dumps(comparison_settings, sort_keys=True).encode("utf-8")).hexdigest()[:16]
RUN_DIR = RESULTS_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)
write_csv(RUN_DIR / "paired_results.csv", paired_records)
write_csv(RUN_DIR / "grid_summary.csv", cell_records)
(RUN_DIR / "settings.json").write_text(json.dumps(comparison_settings, indent=2) + "\n", encoding="utf-8")
(RESULTS_ROOT / "latest_run.json").write_text(
    json.dumps({"run_id": run_id, "run_directory": str(RUN_DIR)}, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved paired records and grid summary to {RUN_DIR}.")


Saved paired records and grid summary to C:\Users\shich\Src\hw1opt\results\05_gaussian_smoothing_vs_robust\ed873a15c5ae8626.


## 4. Smoothing minus RobustCE grid

The displayed number is **CE(Smoothing) - CE(RobustCE)**, averaged over five paired evaluation seeds.

- **Green / positive:** RobustCE is better.
- **Red / negative:** smoothing is better.
- **White:** zero or small absolute difference.
- **Saturation:** absolute magnitude on a symmetric linear scale centered at zero.

The horizontal axis lists RobustCE delta, with its matched epsilon shown below. Each tested eta/delta pair is one equal-size bin. Hover shows both test losses, paired variation, seed wins, selected learning rates, and mean prediction confidence.

Positive differences are green because they mean RobustCE has lower test CE; negative differences are red because smoothing is better. The subtraction order and numerical values are unchanged.


In [6]:
by_cell = {(row["eta"], row["delta"]): row for row in cell_records}
color_grid, text_grid, hover_grid = [], [], []
for eta in ETAS:
    colors, texts, details = [], [], []
    for delta, epsilon in PAIRS:
        row = by_cell[(eta, delta)]
        if not row["complete"]:
            colors.append(None)
            texts.append("missing")
            details.append([eta, delta, epsilon] + [None] * 13)
            continue
        difference = row["difference_mean"]
        colors.append(0.0 if abs(difference) <= WIN_ATOL else difference)
        texts.append(
            "tie" if abs(difference) <= WIN_ATOL
            else f"{difference:+.3f}" if abs(difference) >= 0.001
            else f"{difference:+.1e}"
        )
        details.append([
            eta, delta, epsilon, row["smoothing_test_ce_mean"], row["robust_test_ce_mean"],
            difference, row["relative_difference_percent"], row["difference_std"],
            row["smoothing_seed_wins"], row["robust_seed_wins"], row["paired_seeds"],
            row["smoothing_lr"], row["robust_lr"],
            100 * row["smoothing_confidence_mean"], 100 * row["robust_confidence_mean"],
            row["winner"],
        ])
    color_grid.append(colors)
    text_grid.append(texts)
    hover_grid.append(details)

color_limit = max([abs(value) for row in color_grid for value in row if value is not None] + [WIN_ATOL])
fig = go.Figure(go.Heatmap(
    x=[f"{delta:g}" for delta, _ in PAIRS],
    y=[f"{eta:g}" for eta in ETAS],
    z=color_grid, zmin=-color_limit, zmax=color_limit,
    colorscale=[[0.0, "#991b1b"], [0.5, "#ffffff"], [1.0, "#166534"]],
    text=text_grid, texttemplate="%{text}", textfont=dict(size=10, color="#111827"),
    customdata=hover_grid, xgap=2, ygap=2, hoverongaps=False,
    colorbar=dict(title="CE(S) - CE(R)", thickness=18),
    hovertemplate=(
        "eta=%{customdata[0]}<br>RobustCE delta=%{customdata[1]}<br>Smoothing epsilon=%{customdata[2]}"
        "<br>Mean Smoothing test CE=%{customdata[3]:.8f}"
        "<br>Mean RobustCE test CE=%{customdata[4]:.8f}"
        "<br>Smoothing - RobustCE=%{customdata[5]:+.8f}"
        "<br>Relative difference to RobustCE=%{customdata[6]:+.4f}%"
        "<br>Paired difference sample SD=%{customdata[7]:.8f}"
        "<br>Seed wins: Smoothing=%{customdata[8]}, RobustCE=%{customdata[9]}, paired=%{customdata[10]}"
        "<br>Selected lr: Smoothing=%{customdata[11]}, RobustCE=%{customdata[12]}"
        "<br>Mean confidence: Smoothing=%{customdata[13]:.2f}%, RobustCE=%{customdata[14]:.2f}%"
        "<br>Mean winner=%{customdata[15]}<extra></extra>"
    ),
))
fig.update_xaxes(
    title_text="RobustCE delta; matched smoothing epsilon=2*delta below",
    type="category", tickangle=-35,
    tickvals=[f"{delta:g}" for delta, _ in PAIRS],
    ticktext=[f"{delta:g}<br>eps={epsilon:g}" for delta, epsilon in PAIRS],
)
fig.update_yaxes(
    title_text="Training-label flip probability eta",
    type="category", autorange="reversed",
)
fig.update_layout(
    title="Gaussian: Smoothing - RobustCE<br><sup>Ordinary clean test CE; matched epsilon=2*delta</sup>",
    height=700, margin=dict(l=90, r=190, t=120, b=145), plot_bgcolor="#e5e7eb",
    annotations=[dict(
        x=0, y=1.08, xref="paper", yref="paper", showarrow=False, xanchor="left",
        text="Green: RobustCE better; red: smoothing better; white: near zero.",
    )],
)
fig.show(config={"displaylogo": False, "responsive": True})
fig.write_html(RUN_DIR / "smoothing_minus_robust_heatmap.html", include_plotlyjs=True, full_html=True)


## 5. Reading the result

Compare the two fitted procedures at analytically matched strengths. Matching positive-margin penalties does not guarantee identical average confidence, predictions, or test CE because the losses differ on negative margins and use separately selected learning rates.

The delta = 0, epsilon = 0 column is a shared-CE identity control. At the opposite boundary, delta = 0.5 and epsilon = 1 both have individual-example optimum at zero margin, but finite-budget Adam may leave different residual margins. Inspect learning rates and confidence before attributing a boundary-cell advantage to the worst-case construction.

The grid is exploratory on the existing five matched evaluation seeds. Small differences and seed variation matter. A global claim of superiority would require a frozen comparison on fresh evaluation seeds.


In [7]:
protocol = {
    **comparison_settings, "run_id": run_id,
    "metric": "ordinary CE on clean held-out labels",
    "color_rule": "negative red = Smoothing better; positive green = RobustCE better; near zero white",
    "numerical_tie": f"absolute mean difference <= {WIN_ATOL}",
    "complete_cells": len(complete_cells),
    "total_cells": len(ETAS) * len(PAIRS),
    "paired_records": len(paired_records),
    "mean_winner_counts": counts,
    "new_training_runs": 0,
    "learning_rate_selection": "unchanged separate validation-only selection from source notebooks",
}
(RUN_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
assert len(cell_records) == len(ETAS) * len(PAIRS)
for row in cell_records:
    assert math.isclose(row["epsilon"], 2 * row["delta"], rel_tol=0, abs_tol=1e-12)
for row in complete_cells:
    assert row["smoothing_seed_wins"] + row["robust_seed_wins"] + row["seed_ties"] == len(EVALUATION_SEEDS)
for source in sources.values():
    for filename, checksum in source["file_hashes"].items():
        assert sha256(source["directory"] / filename) == checksum
print(f"Complete: {len(complete_cells)} cells, {len(paired_records)} paired records, no new training.")
print(f"Interactive HTML and CSV/JSON results: {RUN_DIR}")


Complete: 84 cells, 420 paired records, no new training.
Interactive HTML and CSV/JSON results: C:\Users\shich\Src\hw1opt\results\05_gaussian_smoothing_vs_robust\ed873a15c5ae8626
